# Minimal Colab: Export and Upload a Relayered Hugging Face Model

This notebook installs the minimal export dependencies, clones or unpacks this repo, downloads a base model, exports a relayered variant, verifies the layer count, and optionally uploads the result.


In [ ]:
%pip -q install huggingface_hub safetensors uv


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/your-name/RYS.git"
GITHUB_TOKEN = ""
REPO_ARCHIVE_PATH = ""
SOURCE_REPO_ID = "Qwen/Qwen3.5-27B-FP8"
EXPORT_REPO_ID = "your-hf-name/Qwen3.5-27B-FP8-block-30-34"
BLOCK_SPEC = "30,34"
SOURCE_DIR = "/content/models/source-model"
OUTPUT_DIR = "/content/models/exported-model"
DO_UPLOAD = False
HF_TOKEN = ""
repo_dir = Path("/content/RYS")


In [ ]:
import os, shutil, subprocess, tarfile, zipfile

def run(cmd, cwd=None, env=None):
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd, env=env)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if REPO_ARCHIVE_PATH:
    archive_path = Path(REPO_ARCHIVE_PATH)
    if archive_path.suffix == ".zip":
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall("/content")
    else:
        with tarfile.open(archive_path) as tf:
            tf.extractall("/content")
    if not repo_dir.exists():
        extracted = [p for p in Path("/content").iterdir() if p.is_dir() and p.name != "models"]
        if len(extracted) == 1:
            extracted[0].rename(repo_dir)
        else:
            raise FileNotFoundError("Archive extracted, but /content/RYS was not created. Rename the extracted folder or adjust repo_dir.")
else:
    clone_url = REPO_URL
    if GITHUB_TOKEN and clone_url.startswith("https://"):
        clone_url = clone_url.replace("https://", f"https://{GITHUB_TOKEN}@", 1)
    run(["git", "clone", clone_url, str(repo_dir)])

run(["uv", "sync"], cwd=repo_dir)


In [ ]:
run(["python", "-m", "huggingface_hub", "download", SOURCE_REPO_ID, "--local-dir", SOURCE_DIR, "--local-dir-use-symlinks", "False"])


In [ ]:
run(["uv", "run", "python", "-m", "hf_export.export_model", "--source", SOURCE_DIR, "--source-repo-id", SOURCE_REPO_ID, "--output", OUTPUT_DIR, "--blocks", BLOCK_SPEC, "--overwrite"], cwd=repo_dir)


In [ ]:
import json
root = Path(OUTPUT_DIR)
config = json.loads((root / "config.json").read_text())
manifest = json.loads((root / "rys_export_manifest.json").read_text())
print("target layers:", manifest["target_num_layers"])
print("extra layers:", manifest["extra_layers"])
print("spec:", manifest["spec"])
print("config num_hidden_layers:", config.get("num_hidden_layers", config.get("text_config", {}).get("num_hidden_layers")))


In [ ]:
if DO_UPLOAD:
    if not HF_TOKEN:
        raise RuntimeError("Set HF_TOKEN before upload.")
    env = dict(os.environ)
    env["HF_TOKEN"] = HF_TOKEN
    run(["uv", "run", "python", "-m", "hf_export.upload_to_hf", "--folder", OUTPUT_DIR, "--repo-id", EXPORT_REPO_ID], cwd=repo_dir, env=env)
else:
    print("Upload skipped. Set DO_UPLOAD = True to publish the exported model.")
